In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# STAGE 1: TRAIN NEW FROZEN DICTABERT MODEL
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from google.colab import drive
from torch.utils.data import Dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# 1. Mount Google Drive
drive.mount('/content/drive')

MODEL_NAME = "dicta-il/dictabert"
SAVE_PATH = "/content/drive/MyDrive/stage1_dictabert_FROZEN/"
os.makedirs(SAVE_PATH, exist_ok=True)

LABEL_2_ID = {"O": 0, "B-MU": 1, "I-MU": 2}
ID_2_LABEL = {0: "O", 1: "B-MU", 2: "I-MU"}

# ============================================================
# 2. DATASET CLASS
# ============================================================
class GroupedStage1BIODataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len

        df = pd.read_csv(csv_path)
        df = df.dropna(subset=["Full_Raw_Sentence", "MeaningVal"])

        self.grouped_data = []
        for sent, group in df.groupby("Full_Raw_Sentence", sort=False):
            sentence = str(sent).strip()
            mv_spans = []
            search_start_pos = 0

            for _, row in group.iterrows():
                mv = str(row["MeaningVal"]).strip()
                if not mv:
                    continue

                start_pos = sentence.find(mv, search_start_pos)
                if start_pos == -1:
                    start_pos = sentence.find(mv)

                if start_pos != -1:
                    end_pos = start_pos + len(mv)
                    mv_spans.append({"text": mv, "start": start_pos, "end": end_pos})
                    search_start_pos = end_pos

            if mv_spans:
                self.grouped_data.append(
                    {"sentence": sentence, "meaning_spans": mv_spans}
                )

    def __len__(self):
        return len(self.grouped_data)

    def __getitem__(self, idx):
        item = self.grouped_data[idx]
        sentence = item["sentence"]
        meaning_spans = item["meaning_spans"]

        encoding = self.tokenizer(
            sentence,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        offsets = encoding["offset_mapping"].squeeze(0).tolist()
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        valid_offsets = [o for o in offsets if o != [0, 0]]
        last_covered_char = valid_offsets[-1][1] if valid_offsets else 0
        is_truncated = last_covered_char < len(sentence)

        labels = [LABEL_2_ID["O"]] * len(offsets)

        for i, (start_tok, end_tok) in enumerate(offsets):
            if start_tok == 0 and end_tok == 0:
                labels[i] = -100

        sorted_spans = sorted(meaning_spans, key=lambda x: x["start"])

        for span in sorted_spans:
            mv_start = span["start"]
            mv_end = span["end"]

            if is_truncated and mv_start >= last_covered_char:
                continue

            first_token_matched = False

            for idx_tok, (start_tok, end_tok) in enumerate(offsets):
                if labels[idx_tok] == -100:
                    continue

                if not (end_tok <= mv_start or start_tok >= mv_end):
                    if not first_token_matched:
                        labels[idx_tok] = LABEL_2_ID["B-MU"]
                        first_token_matched = True
                    else:
                        labels[idx_tok] = LABEL_2_ID["I-MU"]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(labels, dtype=torch.long),
        }

# ============================================================
# 3. WEIGHTED LOSS TRAINER
# ============================================================
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = (
            torch.tensor(self.class_weights, dtype=torch.float).to(logits.device)
            if self.class_weights
            else None
        )
        loss_fct = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# ============================================================
# 4. LOAD BASE DICTABERT & FREEZE IT
# ============================================================
print("\n📦 Loading Base DictaBERT Tokenizer & Datasets...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = GroupedStage1BIODataset("train_set.csv", tokenizer)
val_dataset = GroupedStage1BIODataset("val_set.csv", tokenizer)

print("\n🧊 Loading FRESH Base DictaBERT Model & Freezing Backbone...")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID_2_LABEL, label2id=LABEL_2_ID
)

# FREEZE DICTABERT TRANSFORMER BODY
for param in model.bert.parameters():
    param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"  • Total Parameters     : {total_params:,}")
print(f"  • Trainable Parameters : {trainable_params:,} (ONLY the top classifier head!)")

# ============================================================
# 5. TRAIN FROZEN MODEL WITH UNWEIGHTED LOSS [1, 1, 1]
# ============================================================
print("\n🚀 Training Frozen Linear Probe Baseline with Unweighted Loss [1.0, 1.0, 1.0]...")

best_weights = [1.0, 1.0, 1.0]

training_args = TrainingArguments(
    output_dir="./tmp_frozen_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    save_total_limit=1,
    report_to="none"
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    class_weights=best_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("\n" + "=" * 65)
print(f"🎉 NEW FROZEN STAGE 1 MODEL TRAINED & SAVED TO:")
print(f" 📁 {SAVE_PATH}")
print("=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📦 Loading Base DictaBERT Tokenizer & Datasets...


config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: reconstructing file:   0%|          |  0.00B / 1.50MB            

vocab.txt: downloading bytes:           |  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 NEW FROZEN STAGE 1 MODEL TRAINED & SAVED TO:
 📁 /content/drive/MyDrive/stage1_dictabert_FROZEN/


In [ ]:
# ============================================================
#  MODEL 1 EVALUATION ON TEST SET
# ============================================================
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report as sklearn_classification_report
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer

DRIVE_SAVE_PATH = (
    "/content/drive/MyDrive/stage1_dictabert_FROZEN/"
)

TEST_CSV = (
    "test_set.csv"
    if os.path.exists("test_set.csv")
    else "stage1_splits/test_set.csv"
)

ID_2_LABEL = {0: "O", 1: "B-MU", 2: "I-MU"}
LABEL_2_ID = {"O": 0, "B-MU": 1, "I-MU": 2}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------
# DATASET CLASS DEFINITION
# ------------------------------------------------------------
class GroupedStage1BIODataset(Dataset):

  def __init__(self, csv_path, tokenizer, max_len=128):
    self.tokenizer = tokenizer
    self.max_len = max_len

    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["Full_Raw_Sentence", "MeaningVal"])

    self.grouped_data = []
    for sent, group in df.groupby("Full_Raw_Sentence", sort=False):
      sentence = str(sent).strip()
      mv_spans = []
      search_start_pos = 0

      for _, row in group.iterrows():
        mv = str(row["MeaningVal"]).strip()
        if not mv:
          continue

        start_pos = sentence.find(mv, search_start_pos)
        if start_pos == -1:
          start_pos = sentence.find(mv)

        if start_pos != -1:
          end_pos = start_pos + len(mv)
          mv_spans.append({"text": mv, "start": start_pos, "end": end_pos})
          search_start_pos = end_pos

      if mv_spans:
        self.grouped_data.append(
            {"sentence": sentence, "meaning_spans": mv_spans}
        )

  def __len__(self):
    return len(self.grouped_data)

  def __getitem__(self, idx):
    item = self.grouped_data[idx]
    sentence = item["sentence"]
    meaning_spans = item["meaning_spans"]

    encoding = self.tokenizer(
        sentence,
        max_length=self.max_len,
        padding="max_length",
        truncation=True,
        return_offsets_mapping=True,
        return_tensors="pt",
    )

    offsets = encoding["offset_mapping"].squeeze(0).tolist()
    input_ids = encoding["input_ids"].squeeze(0)
    attention_mask = encoding["attention_mask"].squeeze(0)

    valid_offsets = [o for o in offsets if o != [0, 0]]
    last_covered_char = valid_offsets[-1][1] if valid_offsets else 0
    is_truncated = last_covered_char < len(sentence)

    labels = [LABEL_2_ID["O"]] * len(offsets)

    for i, (start_tok, end_tok) in enumerate(offsets):
      if start_tok == 0 and end_tok == 0:
        labels[i] = -100

    sorted_spans = sorted(meaning_spans, key=lambda x: x["start"])

    for span in sorted_spans:
      mv_start = span["start"]
      mv_end = span["end"]

      if is_truncated and mv_start >= last_covered_char:
        continue

      first_token_matched = False

      for idx_tok, (start_tok, end_tok) in enumerate(offsets):
        if labels[idx_tok] == -100:
          continue

        if not (end_tok <= mv_start or start_tok >= mv_end):
          if not first_token_matched:
            labels[idx_tok] = LABEL_2_ID["B-MU"]
            first_token_matched = True
          else:
            labels[idx_tok] = LABEL_2_ID["I-MU"]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": torch.tensor(labels, dtype=torch.long),
    }


# ------------------------------------------------------------
# LOAD FROZEN MODEL CHECKPOINT
# ------------------------------------------------------------
print(f"📦 Loading Frozen Stage 1 Model from: {DRIVE_SAVE_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(DRIVE_SAVE_PATH)
model = AutoModelForTokenClassification.from_pretrained(DRIVE_SAVE_PATH).to(
    device
)
model.eval()


# ------------------------------------------------------------
# 1. HELPER FUNCTIONS
# ------------------------------------------------------------
def normalize_hebrew(text):
  text = str(text)
  text = re.sub(r"[^\w\s]", "", text)
  return " ".join(text.strip().split())


def compute_word_iou(pred_phrase, gold_phrase):
  pred_words = normalize_hebrew(pred_phrase).split()
  gold_words = normalize_hebrew(gold_phrase).split()

  if not pred_words or not gold_words:
    return 1.0 if pred_words == gold_words else 0.0

  p_counter, g_counter = Counter(pred_words), Counter(gold_words)
  intersection = sum((p_counter & g_counter).values())
  union = sum((p_counter | g_counter).values())

  return intersection / union if union > 0 else 0.0


def extract_spans_from_bio(token_ids, pred_ids, offsets, sentence, tokenizer):
  spans = []
  current_tokens = []

  for token_id, p_id, (start_char, end_char) in zip(
      token_ids, pred_ids, offsets
  ):
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    if token in tokenizer.all_special_tokens or (
        start_char == 0 and end_char == 0
    ):
      if current_tokens:
        spans.append(current_tokens)
        current_tokens = []
      continue

    if p_id == 1:  # B-MU
      if current_tokens:
        spans.append(current_tokens)
      current_tokens = [(start_char, end_char)]
    elif p_id == 2:  # I-MU
      if current_tokens:
        current_tokens.append((start_char, end_char))
      else:
        current_tokens = [(start_char, end_char)]
    else:  # O
      if current_tokens:
        spans.append(current_tokens)
        current_tokens = []

  if current_tokens:
    spans.append(current_tokens)

  extracted_mvs = []
  for span in spans:
    min_start = span[0][0]
    max_end = span[-1][1]
    phrase = sentence[min_start:max_end].strip()
    phrase_clean = normalize_hebrew(phrase)
    if phrase_clean:
      extracted_mvs.append(phrase_clean)
  return extracted_mvs


def bio_tags_to_spans(bio_seq):
  """Converts a sequence of BIO tags into explicit entity spans (start_idx, end_idx, label)."""
  spans = []
  start = None

  for i, tag in enumerate(bio_seq):
    if tag == "B-MU":
      if start is not None:
        spans.append((start, i - 1, "MU"))
      start = i
    elif tag == "I-MU":
      if start is None:
        start = i
    else:  # 'O' or -100
      if start is not None:
        spans.append((start, i - 1, "MU"))
        start = None

  if start is not None:
    spans.append((start, len(bio_seq) - 1, "MU"))

  return set(spans)


def compute_strict_span_f1(all_gold_bio, all_pred_bio):
  """Calculates strict Span Precision, Recall, and F1 natively without seqeval."""
  tp, fp, fn = 0, 0, 0

  for gold_seq, pred_seq in zip(all_gold_bio, all_pred_bio):
    gold_spans = bio_tags_to_spans(gold_seq)
    pred_spans = bio_tags_to_spans(pred_seq)

    tp += len(gold_spans & pred_spans)
    fp += len(pred_spans - gold_spans)
    fn += len(gold_spans - pred_spans)

  precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
  recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
  f1 = (
      2 * (precision * recall) / (precision + recall)
      if (precision + recall) > 0
      else 0.0
  )

  return precision, recall, f1, tp, fp, fn


# ------------------------------------------------------------
# 2. RUN TOKEN & SPAN EVALUATION
# ------------------------------------------------------------
test_dataset = GroupedStage1BIODataset(TEST_CSV, tokenizer)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=16, shuffle=False
)

flat_token_preds = []
flat_token_labels = []
raw_pred_seqs = []
raw_gold_seqs = []

with torch.no_grad():
  for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].numpy()

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    preds = np.argmax(outputs.logits.cpu().numpy(), axis=2)

    for p_row, l_row in zip(preds, labels):
      c_preds, c_labs = [], []
      for p, l in zip(p_row, l_row):
        if l != -100:
          flat_token_preds.append(p)
          flat_token_labels.append(l)
          c_preds.append(ID_2_LABEL[p])
          c_labs.append(ID_2_LABEL[l])
      raw_pred_seqs.append(c_preds)
      raw_gold_seqs.append(c_labs)

print("\n" + "=" * 65)
print(f"1️⃣ TOKEN-LEVEL CLASSIFICATION REPORT (TEST SET: {TEST_CSV})")
print("=" * 65)
print(
    sklearn_classification_report(
        flat_token_labels,
        flat_token_preds,
        target_names=["O", "B-MU", "I-MU"],
        digits=4,
        zero_division=0,
    )
)

prec, rec, span_f1, tp, fp, fn = compute_strict_span_f1(
    raw_gold_seqs, raw_pred_seqs
)

print("\n" + "=" * 65)
print("2️⃣ STRICT SPAN-LEVEL REPORT (NATIVE PHRASE MATCHING)")
print("=" * 65)
print(f" • True Positives (Exact Spans)  : {tp}")
print(f" • False Positives (Extra Spans) : {fp}")
print(f" • False Negatives (Missed Spans): {fn}")
print(f" • Strict Span Precision         : {prec * 100:.2f}%")
print(f" • Strict Span Recall            : {rec * 100:.2f}%")
print(f" • Strict Span F1-Score          : {span_f1 * 100:.2f}%")

# ------------------------------------------------------------
# 3. RUN IOU & SENTENCE EXACT MATCH EVALUATION
# ------------------------------------------------------------
test_df = pd.read_csv(TEST_CSV)
grouped = test_df.groupby("Full_Raw_Sentence", sort=False)

sentence_em_correct = 0
total_sentences = 0
all_span_ious = []

for sentence_raw, group in grouped:
  sentence = str(sentence_raw).strip()
  gold_mvs = [
      normalize_hebrew(x)
      for x in group["MeaningVal"].dropna().astype(str).tolist()
  ]

  inputs = tokenizer(
      sentence,
      return_tensors="pt",
      truncation=True,
      max_length=128,
      return_offsets_mapping=True,
  ).to(device)

  offsets = inputs.pop("offset_mapping")[0].cpu().tolist()

  with torch.no_grad():
    outputs = model(**inputs)

  pred_ids = torch.argmax(outputs.logits, dim=2)[0].cpu().tolist()
  token_ids = inputs["input_ids"][0].cpu().tolist()

  pred_mvs = extract_spans_from_bio(
      token_ids, pred_ids, offsets, sentence, tokenizer
  )

  # Calculate Sentence-Level Exact Set Match
  if set(pred_mvs) == set(gold_mvs):
    sentence_em_correct += 1
  total_sentences += 1

  # Calculate Word IoU per Gold Meaning Value
  for gold_mv in gold_mvs:
    best_mv_iou = 0.0
    for pred_mv in pred_mvs:
      iou = compute_word_iou(pred_mv, gold_mv)
      if iou > best_mv_iou:
        best_mv_iou = iou
    all_span_ious.append(best_mv_iou)

avg_iou = np.mean(all_span_ious) * 100 if all_span_ious else 0.0
sentence_em_acc = (sentence_em_correct / max(total_sentences, 1)) * 100

print("\n" + "=" * 65)
print("3️⃣ ADVANCED OVERLAP METRICS (IOU & SENTENCE MATCH)")
print("=" * 65)
print(f" • Mean Phrase Word-Level IoU : {avg_iou:.2f}%")
print(
    f" • Sentence Exact Set Match   : {sentence_em_acc:.2f}%"
    f" ({sentence_em_correct}/{total_sentences})"
)
print("=" * 65)

📦 Loading Frozen Stage 1 Model from: /content/drive/MyDrive/stage1_dictabert_FROZEN/...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


1️⃣ TOKEN-LEVEL CLASSIFICATION REPORT (TEST SET: test_set.csv)
              precision    recall  f1-score   support

           O     0.7762    0.6060    0.6806      2358
        B-MU     0.7364    0.8405    0.7850      3673
        I-MU     0.6821    0.6815    0.6818      2223

    accuracy                         0.7307      8254
   macro avg     0.7316    0.7093    0.7158      8254
weighted avg     0.7332    0.7307    0.7274      8254


2️⃣ STRICT SPAN-LEVEL REPORT (NATIVE PHRASE MATCHING)
 • True Positives (Exact Spans)  : 2520
 • False Positives (Extra Spans) : 1975
 • False Negatives (Missed Spans): 1153
 • Strict Span Precision         : 56.06%
 • Strict Span Recall            : 68.61%
 • Strict Span F1-Score          : 61.70%

3️⃣ ADVANCED OVERLAP METRICS (IOU & SENTENCE MATCH)
 • Mean Phrase Word-Level IoU : 77.84%
 • Sentence Exact Set Match   : 29.52% (235/796)


Stage 2

In [ ]:
# ============================================================
# STAGE 2: TRAIN NEW FROZEN DICTABERT MODEL
# ============================================================
import os
import re
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from torch.utils.data import Dataset
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# 1. Mount Google Drive
drive.mount('/content/drive')

MODEL_NAME = "dicta-il/dictabert"
SAVE_PATH = "/content/drive/MyDrive/Arbel_Workshop/models/stage2_dictabert_FROZEN/"
os.makedirs(SAVE_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def normalize_hebrew(text):
    text = str(text)
    text = re.sub(r"[^\w\s]", "", text)
    return " ".join(text.strip().split())

# ============================================================
# 2. DATASET CLASS FOR EXTRACTIVE QA
# ============================================================
class Stage2QADataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=160):
        self.tokenizer = tokenizer
        self.max_len = max_len

        df = pd.read_csv(csv_path)
        df = df.dropna(subset=["Full_Raw_Sentence", "MeaningVal", "Referent"])

        self.samples = []
        for _, row in df.iterrows():
            sentence = str(row["Full_Raw_Sentence"]).strip()
            mv = str(row["MeaningVal"]).strip()
            referent = str(row["Referent"]).strip()
            prompt_word = str(row.get("Prompt_Word", mv)).strip()

            if not mv or not referent:
                continue

            marked_sentence = sentence.replace(mv, f"<mv> {mv} </mv>", 1)
            context = f"גירוי: {prompt_word} | משפט: {marked_sentence}"
            question = f"מהו הרפרנט של יחידת המשמעות: {mv} ?"

            self.samples.append({
                "question": question,
                "context": context,
                "referent": referent,
                "prompt_word": prompt_word
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        context = item["context"]
        question = item["question"]
        referent = item["referent"]
        prompt_word = item["prompt_word"]

        # Locate character positions of referent in context
        ref_start_char = context.find(referent)
        if ref_start_char == -1:
            ref_start_char = context.find(prompt_word)

        if ref_start_char != -1:
            ref_end_char = ref_start_char + len(referent)
        else:
            ref_start_char = 0
            ref_end_char = 0

        encoding = self.tokenizer(
            question,
            context,
            max_length=self.max_len,
            padding="max_length",
            truncation="only_second",
            return_offsets_mapping=True,
            return_token_type_ids=True,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        token_type_ids = encoding["token_type_ids"].squeeze(0)
        offsets = encoding["offset_mapping"].squeeze(0).tolist()

        # Convert character positions to token positions
        start_token_idx = 0
        end_token_idx = 0

        if ref_start_char != 0 or ref_end_char != 0:
            for i, (tok_start, tok_end) in enumerate(offsets):
                if token_type_ids[i] == 1:
                    if tok_start <= ref_start_char < tok_end:
                        start_token_idx = i
                    if tok_start < ref_end_char <= tok_end:
                        end_token_idx = i

            if start_token_idx > 0 and end_token_idx == 0:
                end_token_idx = start_token_idx

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "start_positions": torch.tensor(start_token_idx, dtype=torch.long),
            "end_positions": torch.tensor(end_token_idx, dtype=torch.long)
        }

# ============================================================
# 3. LOAD BASE DICTABERT QA MODEL & FREEZE IT
# ============================================================
print("\n📦 Loading Tokenizer and Datasets...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Stage2QADataset("train_set.csv", tokenizer)
val_dataset = Stage2QADataset("val_set.csv", tokenizer)

print("\n🧊 Loading Base DictaBERT QA Model & Freezing Transformer Body...")
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

# FREEZE DICTABERT TRANSFORMER BODY
for param in model.bert.parameters():
    param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"  • Total Parameters     : {total_params:,}")
print(f"  • Trainable Parameters : {trainable_params:,} (ONLY start/end QA heads!)")

# ============================================================
# 4. TRAIN FROZEN BASELINE MODEL
# ============================================================
print("\n🚀 Training Frozen Stage 2 Baseline...")

training_args = TrainingArguments(
    output_dir="./tmp_frozen_s2_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("\n" + "=" * 65)
print(f"🎉 FROZEN STAGE 2 MODEL TRAINED & SAVED TO:")
print(f" 📁 {SAVE_PATH}")
print("=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📦 Loading Tokenizer and Datasets...

🧊 Loading Base DictaBERT QA Model & Freezing Transformer Body...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  • Total Parameters     : 183,756,290
  • Trainable Parameters : 1,538 (ONLY start/end QA heads!)

🚀 Training Frozen Stage 2 Baseline...


Epoch,Training Loss,Validation Loss
1,1.539646,1.421886
2,1.485862,1.361948
3,1.437908,1.341035
4,1.388112,1.338264
5,1.475852,1.320598
6,1.548371,1.313866
7,1.441117,1.313594
8,1.532907,1.313982
9,1.499241,1.312619
10,1.511773,1.311017


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 FROZEN STAGE 2 MODEL TRAINED & SAVED TO:
 📁 /content/drive/MyDrive/Arbel_Workshop/models/stage2_dictabert_FROZEN/


In [ ]:
# ============================================================
# EVALUATION FOR FROZEN STAGE 2
# ============================================================
import os
import re
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

DRIVE_SAVE_PATH = "/content/drive/MyDrive/Arbel_Workshop/models/stage2_dictabert_FROZEN/"

TEST_CSV = "test_set.csv" if os.path.exists("test_set.csv") else "stage2_splits/test_set.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"📦 Loading Frozen Stage 2 Model from: {DRIVE_SAVE_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(DRIVE_SAVE_PATH)
model = AutoModelForQuestionAnswering.from_pretrained(DRIVE_SAVE_PATH).to(device)
model.eval()

# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------
def normalize_hebrew(text):
    text = str(text)
    text = re.sub(r"[^\w\s]", "", text)
    return " ".join(text.strip().split())

def compute_token_f1(prediction, gold):
    pred_tokens = normalize_hebrew(prediction).split()
    gold_tokens = normalize_hebrew(gold).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    pred_counts = pd.Series(pred_tokens).value_counts().to_dict()
    gold_counts = pd.Series(gold_tokens).value_counts().to_dict()

    common = sum(min(pred_counts.get(tok, 0), gold_counts.get(tok, 0)) for tok in pred_counts)

    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

# ------------------------------------------------------------
# RUN INFERENCE
# ------------------------------------------------------------
test_df = pd.read_csv(TEST_CSV).dropna(subset=["Full_Raw_Sentence", "MeaningVal", "Referent"])

s2_exact_correct = 0
s2_total = 0
s2_f1_scores = []

for _, row in test_df.iterrows():
    sentence = str(row["Full_Raw_Sentence"]).strip()
    mv = str(row["MeaningVal"]).strip()
    gold_ref = normalize_hebrew(row["Referent"])
    prompt_word = str(row.get("Prompt_Word", mv)).strip()

    marked_sentence = sentence.replace(mv, f"<mv> {mv} </mv>", 1)
    context = f"גירוי: {prompt_word} | משפט: {marked_sentence}"
    question = f"מהו הרפרנט של יחידת המשמעות: {mv} ?"

    encoding = tokenizer(
        question,
        context,
        max_length=160,
        truncation="only_second",
        padding="max_length",
        return_tensors="pt"
    ).to(device)

    token_type_ids = encoding.get("token_type_ids")

    with torch.no_grad():
        outputs = model(**encoding)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    # Mask out Question tokens
    if token_type_ids is not None:
        context_mask = (token_type_ids[0] == 1)
        start_logits = start_logits.masked_fill(~context_mask, -1e9)
        end_logits = end_logits.masked_fill(~context_mask, -1e9)

    start_pred = torch.argmax(start_logits).item()
    end_pred = torch.argmax(end_logits).item()

    input_ids = encoding["input_ids"][0]

    if start_pred <= end_pred and end_pred < len(input_ids):
        pred_ids = input_ids[start_pred : end_pred + 1]
        pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True)
        pred_text = re.sub(r"</?mv>", "", pred_text)
    else:
        pred_text = ""

    pred_text_clean = normalize_hebrew(pred_text)

    if pred_text_clean == gold_ref:
        s2_exact_correct += 1

    s2_f1_scores.append(compute_token_f1(pred_text_clean, gold_ref))
    s2_total += 1

exact_match_acc = (s2_exact_correct / max(s2_total, 1)) * 100
mean_token_f1 = np.mean(s2_f1_scores) * 100 if s2_f1_scores else 0.0

print("\n" + "=" * 65)
print(f"STAGE 2 FROZEN BASELINE EVALUATION RESULTS (TEST SET: {TEST_CSV})")
print("=" * 65)
print(f" • Exact Match Accuracy : {exact_match_acc:.2f}% ({s2_exact_correct}/{s2_total})")
print(f" • Token-Level F1 Score : {mean_token_f1:.2f}%")
print("=" * 65)

📦 Loading Frozen Stage 2 Model from: /content/drive/MyDrive/Arbel_Workshop/models/stage2_dictabert_FROZEN/...


Loading weights:   0%|          | 0/199 [00:04<?, ?it/s]


STAGE 2 FROZEN BASELINE EVALUATION RESULTS (TEST SET: test_set.csv)
 • Exact Match Accuracy : 55.23% (2272/4114)
 • Token-Level F1 Score : 58.00%


Stage 3

In [ ]:
# ============================================================
# STAGE 3: TRAIN FROZEN DICTABERT BASELINE
# ============================================================
import json
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

MODEL_NAME = "dicta-il/dictabert"
SAVE_PATH = "/content/drive/MyDrive/Arbel_Workshop/models/stage3_dictabert_FROZEN/"
os.makedirs(SAVE_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STAGE3_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]


# ============================================================
# 2. MEAN POOLING & MODEL ARCHITECTURE
# ============================================================
def mean_pooling(model_output, attention_mask):
  token_embeddings = model_output.last_hidden_state
  input_mask_expanded = (
      attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
  )
  sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
  sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
  return sum_embeddings / sum_mask


class DictaBertMultiTaskFrozen(nn.Module):

  def __init__(self, model_name, num_classes_dict):
    super().__init__()
    self.bert = AutoModel.from_pretrained(model_name)

    # FREEZE DICTABERT BACKBONE
    for param in self.bert.parameters():
      param.requires_grad = False

    hidden_size = self.bert.config.hidden_size
    self.dropout = nn.Dropout(self.bert.config.hidden_dropout_prob)
    self.heads = nn.ModuleDict({
        col: nn.Linear(hidden_size, num_classes_dict[col]) for col in STAGE3_COLS
    })

  def forward(self, input_ids, attention_mask):
    with torch.no_grad():
      outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
      pooled_output = mean_pooling(outputs, attention_mask)

    pooled_output = self.dropout(pooled_output)
    return {col: self.heads[col](pooled_output) for col in STAGE3_COLS}


# ============================================================
# 3. DATASET & ENCODERS
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_df = pd.read_csv("train_set.csv").dropna(
    subset=["Full_Raw_Sentence", "MeaningVal", "Referent"]
)
val_df = pd.read_csv("val_set.csv").dropna(
    subset=["Full_Raw_Sentence", "MeaningVal", "Referent"]
)

encoders = {}
for col in STAGE3_COLS:
  unique_vals = train_df[col].dropna().astype(str).str.strip().unique()
  encoders[col] = {val: i for i, val in enumerate(sorted(unique_vals))}

num_classes_dict = {col: len(encoders[col]) for col in STAGE3_COLS}

# Save Label Encoders immediately
encoder_save_file = os.path.join(SAVE_PATH, "label_encoders.json")
with open(encoder_save_file, "w", encoding="utf-8") as f:
  json.dump(encoders, f, ensure_ascii=False, indent=2)
print(f"✅ Saved label_encoders.json to: {encoder_save_file}")


class Stage3MultiTaskDataset(Dataset):

  def __init__(self, df, tokenizer, encoders, max_len=128):
    self.df = df.reset_index(drop=True)
    self.tokenizer = tokenizer
    self.encoders = encoders
    self.max_len = max_len

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    text = f"משפט: {row['Full_Raw_Sentence']} | יחידה: {row['MeaningVal']} | רפרנט: {row['Referent']}"

    encoding = self.tokenizer(
        text,
        max_length=self.max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    labels = {}
    for col in STAGE3_COLS:
      val = row[col]
      if pd.isna(val) or str(val).strip() not in self.encoders[col]:
        labels[col] = -100
      else:
        labels[col] = self.encoders[col][str(val).strip()]

    return {
        "input_ids": encoding["input_ids"].squeeze(0),
        "attention_mask": encoding["attention_mask"].squeeze(0),
        "labels": {
            col: torch.tensor(labels[col], dtype=torch.long)
            for col in STAGE3_COLS
        },
    }


train_dataset = Stage3MultiTaskDataset(train_df, tokenizer, encoders)
val_dataset = Stage3MultiTaskDataset(val_df, tokenizer, encoders)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# ============================================================
# 4. TRAINING LOOP
# ============================================================
model = DictaBertMultiTaskFrozen(MODEL_NAME, num_classes_dict).to(device)

optimizer = AdamW(model.heads.parameters(), lr=3e-4, weight_decay=0.01)
epochs = 10
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)

loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print("\n🚀 Training Frozen Stage 3 Linear Probe Baseline...")
best_val_loss = float("inf")
weights_save_file = os.path.join(SAVE_PATH, "pytorch_model.bin")

for epoch in range(epochs):
  model.train()
  train_loss = 0.0

  for batch in train_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = {col: batch["labels"][col].to(device) for col in STAGE3_COLS}

    optimizer.zero_grad()
    logits = model(input_ids, attention_mask)

    batch_loss = torch.tensor(0.0, device=device)
    active_heads = 0

    for col in STAGE3_COLS:
      if (labels[col] != -100).any():
        h_loss = loss_fn(logits[col], labels[col])
        if not torch.isnan(h_loss):
          batch_loss += h_loss
          active_heads += 1

    if active_heads > 0:
      batch_loss.backward()
      torch.nn.utils.clip_grad_norm_(model.heads.parameters(), max_norm=1.0)
      optimizer.step()
      scheduler.step()
      train_loss += batch_loss.item()

  # Validation Check
  model.eval()
  val_loss = 0.0
  valid_batches = 0
  with torch.no_grad():
    for batch in val_loader:
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = {col: batch["labels"][col].to(device) for col in STAGE3_COLS}
      logits = model(input_ids, attention_mask)

      for col in STAGE3_COLS:
        if (labels[col] != -100).any():
          h_loss = loss_fn(logits[col], labels[col])
          if not torch.isnan(h_loss):
            val_loss += h_loss.item()
            valid_batches += 1

  avg_train_loss = train_loss / max(len(train_loader), 1)
  avg_val_loss = val_loss / max(valid_batches, 1)
  print(
      f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val"
      f" Loss: {avg_val_loss:.4f}"
  )

  if avg_val_loss < best_val_loss and not np.isnan(avg_val_loss):
    best_val_loss = avg_val_loss
    torch.save(model.state_dict(), weights_save_file)
    print(f"  💾 Saved improved model weights to: {weights_save_file}")

tokenizer.save_pretrained(SAVE_PATH)

if not os.path.exists(weights_save_file):
  print("⚠️ Saving final epoch model weights as fallback...")
  torch.save(model.state_dict(), weights_save_file)

print("\n" + "=" * 65)
print(f"🎉 FROZEN STAGE 3 MODEL SUCCESSFULLY TRAINED & SAVED TO:")
print(f" 📁 {SAVE_PATH}")
print("=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: reconstructing file:   0%|          |  0.00B / 1.50MB            

vocab.txt: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 Training Frozen Stage 3 Linear Probe Baseline...
Epoch 01/10 | Train Loss: 8.5809 | Val Loss: 1.0923
  💾 Saved improved model weights to: /content/drive/MyDrive/Arbel_Workshop/models/stage3_dictabert_FROZEN/pytorch_model.bin
Epoch 02/10 | Train Loss: 6.4610 | Val Loss: 0.9954
  💾 Saved improved model weights to: /content/drive/MyDrive/Arbel_Workshop/models/stage3_dictabert_FROZEN/pytorch_model.bin
Epoch 03/10 | Train Loss: 6.0241 | Val Loss: 0.9485
  💾 Saved improved model weights to: /content/drive/MyDrive/Arbel_Workshop/models/stage3_dictabert_FROZEN/pytorch_model.bin
Epoch 04/10 | Train Loss: 5.7661 | Val Loss: 0.9220
  💾 Saved improved model weights to: /content/drive/MyDrive/Arbel_Workshop/models/stage3_dictabert_FROZEN/pytorch_model.bin
Epoch 05/10 | Train Loss: 5.6200 | Val Loss: 0.9056
  💾 Saved improved model weights to: /content/drive/MyDrive/Arbel_Workshop/models/stage3_dictabert_FROZEN/pytorch_model.bin
Epoch 06/10 | Train Loss: 5.4980 | Val Loss: 0.8905
  💾 Saved improv

In [ ]:
# ============================================================
# EVALUATION FOR FROZEN STAGE 3
# ============================================================
import json
import os
import re
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

DRIVE_SAVE_PATH = "/content/drive/MyDrive/Arbel_Workshop/models/stage3_dictabert_FROZEN/"
TEST_CSV = (
    "test_set.csv"
    if os.path.exists("test_set.csv")
    else "stage3_splits/test_set.csv"
)
MODEL_NAME = "dicta-il/dictabert"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STAGE3_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]


# ------------------------------------------------------------
# 1. ARCHITECTURE DEFINITION
# ------------------------------------------------------------
def mean_pooling(model_output, attention_mask):
  token_embeddings = model_output.last_hidden_state
  input_mask_expanded = (
      attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
  )
  sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
  sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
  return sum_embeddings / sum_mask


class DictaBertMultiTaskFrozen(nn.Module):

  def __init__(self, model_name, num_classes_dict):
    super().__init__()
    self.bert = AutoModel.from_pretrained(model_name)
    hidden_size = self.bert.config.hidden_size
    self.dropout = nn.Dropout(self.bert.config.hidden_dropout_prob)
    self.heads = nn.ModuleDict({
        col: nn.Linear(hidden_size, num_classes_dict[col]) for col in STAGE3_COLS
    })

  def forward(self, input_ids, attention_mask):
    outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
    pooled_output = mean_pooling(outputs, attention_mask)
    pooled_output = self.dropout(pooled_output)
    return {col: self.heads[col](pooled_output) for col in STAGE3_COLS}


# ------------------------------------------------------------
# 2. SANITY & INTEGRITY CHECKS
# ------------------------------------------------------------
print("🔍 Running Initial Model & Environment Integrity Checks...\n")

# Check 1: Existence of files
encoder_path = os.path.join(DRIVE_SAVE_PATH, "label_encoders.json")
weights_path = os.path.join(DRIVE_SAVE_PATH, "pytorch_model.bin")

if not os.path.exists(encoder_path) or not os.path.exists(weights_path):
  raise FileNotFoundError(
      f"❌ Missing critical files in {DRIVE_SAVE_PATH}.\n"
      "Make sure training completed successfully and saved both"
      " `label_encoders.json` and `pytorch_model.bin`."
  )

with open(encoder_path, "r", encoding="utf-8") as f:
  encoders = json.load(f)

num_classes_dict = {col: len(encoders[col]) for col in STAGE3_COLS}

# Load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DictaBertMultiTaskFrozen(MODEL_NAME, num_classes_dict)
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model = model.to(device).eval()

# Check 2: Inspect weights for NaN or Inf corruption
nan_found = False
for param_name, param_tensor in model.named_parameters():
  if torch.isnan(param_tensor).any() or torch.isinf(param_tensor).any():
    print(f"❌ ERROR: Parameter '{param_name}' contains NaN or Inf values!")
    nan_found = True

if nan_found:
  raise ValueError(
      "❌ Model checkpoint is corrupted with NaN weights! Re-run training with"
      " safe loss logic."
  )
else:
  print("✅ Check 1 Passed: No NaN or Inf parameters detected in checkpoint.")

# Check 3: Forward-pass sanity test on dummy input
dummy_text = "משפט: בוקר טוב | יחידה: בוקר | רפרנט: יום"
dummy_enc = tokenizer(
    dummy_text, return_tensors="pt", padding=True, max_length=128
).to(device)

with torch.no_grad():
  dummy_out = model(dummy_enc["input_ids"], dummy_enc["attention_mask"])

for col in STAGE3_COLS:
  if torch.isnan(dummy_out[col]).any():
    raise ValueError(
        f"❌ Check 2 Failed: Model produced NaN logits on head '{col}'!"
    )

print(
    "✅ Check 2 Passed: Forward pass verified. All 7 linear heads output valid"
    " non-NaN predictions."
)
print("------------------------------------------------------------\n")

# ------------------------------------------------------------
# 3. RUN INFERENCE & EVALUATION ON TEST SET
# ------------------------------------------------------------
print(f"📦 Evaluating Frozen Stage 3 Model on: {TEST_CSV}...\n")
test_df = pd.read_csv(TEST_CSV).dropna(
    subset=["Full_Raw_Sentence", "MeaningVal", "Referent"]
)

y_true = {col: [] for col in STAGE3_COLS}
y_pred = {col: [] for col in STAGE3_COLS}

for _, row in test_df.iterrows():
  text = f"משפט: {row['Full_Raw_Sentence']} | יחידה: {row['MeaningVal']} | רפרנט: {row['Referent']}"
  encoding = tokenizer(
      text,
      max_length=128,
      padding="max_length",
      truncation=True,
      return_tensors="pt",
  ).to(device)

  with torch.no_grad():
    logits = model(encoding["input_ids"], encoding["attention_mask"])

  for col in STAGE3_COLS:
    gold_val = row[col]
    if not pd.isna(gold_val):
      gold_str = str(gold_val).strip()
      if gold_str in encoders[col]:
        pred_id = torch.argmax(logits[col], dim=-1).item()
        gold_id = encoders[col][gold_str]
        y_true[col].append(gold_id)
        y_pred[col].append(pred_id)

# ------------------------------------------------------------
# 4. PRINT DETAILED METRICS
# ------------------------------------------------------------
print("=" * 65)
print("STAGE 3 FROZEN TEST EVALUATION RESULTS")
print("=" * 65)

accs, macro_f1s = [], []

for col in STAGE3_COLS:
  if len(y_true[col]) == 0:
    print(f" ⚠️ {col:10s} | No valid test samples evaluated.")
    continue

  acc = accuracy_score(y_true[col], y_pred[col]) * 100
  f1 = f1_score(y_true[col], y_pred[col], average="macro", zero_division=0) * 100
  accs.append(acc)
  macro_f1s.append(f1)
  print(
      f" • {col:10s} | Accuracy: {acc:6.2f}% | Macro-F1: {f1:6.2f}%"
      f" ({len(y_true[col])} samples)"
  )

print("-" * 65)
if accs:
  print(f" 📊 Average Accuracy across all 7 heads : {np.mean(accs):.2f}%")
  print(f" 📊 Average Macro-F1 across all 7 heads : {np.mean(macro_f1s):.2f}%")
else:
  print("❌ Evaluation failed: No samples were scored.")
print("=" * 65)

🔍 Running Initial Model & Environment Integrity Checks...



config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: reconstructing file:   0%|          |  0.00B / 1.50MB            

vocab.txt: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Check 1 Passed: No NaN or Inf parameters detected in checkpoint.
✅ Check 2 Passed: Forward pass verified. All 7 linear heads output valid non-NaN predictions.
------------------------------------------------------------

📦 Evaluating Frozen Stage 3 Model on: test_set.csv...



/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2356: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


STAGE 3 FROZEN TEST EVALUATION RESULTS
 • dim        | Accuracy:  40.99% | Macro-F1:  25.37% (4108 samples)
 • tr         | Accuracy:  71.30% | Macro-F1:  17.30% (4108 samples)
 • fr         | Accuracy:  85.00% | Macro-F1:  11.39% (4107 samples)
 • sr         | Accuracy:  79.41% | Macro-F1:  18.48% (4114 samples)
 • fe         | Accuracy: 100.00% | Macro-F1: 100.00% (4111 samples)
 • refLevel   | Accuracy:  51.82% | Macro-F1:  29.59% (4112 samples)
 • ss         | Accuracy:  70.51% | Macro-F1:  42.01% (78 samples)
-----------------------------------------------------------------
 📊 Average Accuracy across all 7 heads : 71.29%
 📊 Average Macro-F1 across all 7 heads : 34.88%
